In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import plotly.express as px
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier, plot_tree


df  = pd.read_csv("data/customer_invoices.zip", encoding="ISO-8859-1")

display(Markdown("### Estructura y consistencia de los datos"))
info_df = pd.DataFrame({
    "Tipo de dato": df.dtypes,
    "Registros no nulos": df.notnull().sum(),
    "Valores faltantes (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
display(info_df)

display(Markdown("-" * 70))

display(Markdown("### Vista previa del histórico"))
display(df.head(10))

### Estructura y consistencia de los datos

,Tipo de dato,Registros no nulos,Valores faltantes (%)
InvoiceNo,object,541909,0.00
StockCode,object,541909,0.00
Description,object,540455,0.27
Quantity,int64,541909,0.00
InvoiceDate,object,541909,0.00
UnitPrice,float64,541909,0.00
CustomerID,float64,406829,24.93
Country,object,541909,0.00


----------------------------------------------------------------------

### Vista previa del histórico

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


#### Definición de la variable objetivo: categórica ordinal de 3 clases
Para modelar de forma pragmática el abandono en un entorno transaccional no contractual (donde el cliente no cancela explícitamente una suscripción, sino que simplemente deja de comprar), se aplicó una segmentación temporal estricta para evitar la filtración de datos (data leakage), utilizando los primeros 9 meses como ventana de observación del comportamiento, medimos el período de inactividad (recencia) de los clientes en los últimos 3 meses del histórico. En lugar de una clasificación binaria, se implementa un análisis estadístico de cuantiles para estratificar el riesgo en tres ventanas dinámicas de decisión: Clase 0 (Sin Riesgo / Activo) para clientes dentro del percentil 75 de actividad normal; Clase 1 (Riesgo Moderado / Enfriándose) entre los percentiles 75 y 90, actuando como una ventana de alerta temprana para el CRM; y Clase 2 (Alto Riesgo / Churn Técnico) para aquellos que superan el percentil 90, donde la probabilidad de retorno orgánico es estadísticamente inferior al 10%.

In [ ]:
df = df.dropna(subset=["Description", "CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

date_cut = pd.to_datetime("2011-09-01")

df_obs = df[df["InvoiceDate"] < date_cut].copy()
df_pred = df[df["InvoiceDate"] >= date_cut].copy()

max_date = df["InvoiceDate"].max()

last_contact = df_pred.groupby("CustomerID")["InvoiceDate"].max().reset_index()
last_contact.columns = ["CustomerID", "LastDatePred"]
last_contact["RecencyTarget"] = (max_date - last_contact["LastDatePred"]).dt.days

customers_to_retain = pd.DataFrame({"CustomerID": df_obs["CustomerID"].unique()})

target_df = pd.merge(customers_to_retain, last_contact, on="CustomerID", how="left")

last_date_obs = df_obs.groupby("CustomerID")["InvoiceDate"].max().reset_index()
last_date_obs.columns = ["CustomerID", "LastDateObs"]
target_df = pd.merge(target_df, last_date_obs, on="CustomerID", how="left")

target_df["RecencyTarget"] = target_df["RecencyTarget"].fillna((max_date - target_df["LastDateObs"]).dt.days)

p75 = target_df["RecencyTarget"].quantile(0.75)
p90 = target_df["RecencyTarget"].quantile(0.90)

def assign_class(recency):
    if recency <= p75:
        return 0
    elif recency <= p90:
        return 1
    else:
        return 2

target_df["Target"] = target_df["RecencyTarget"].apply(assign_class)

print(f"Total de registros limpios: {df.shape[0]}")
print(f"Rango de fechas completo: {df["InvoiceDate"].min()} hasta {df["InvoiceDate"].max()}")
display(Markdown("\n #### --- División temporal estratégica ---"))
print(f"Ventana de observación (Features): {df_obs["InvoiceDate"].min()} a {df_obs["InvoiceDate"].max()}")
print(f"Ventana de predicción (Target):   {df_pred["InvoiceDate"].min()} a {df_pred["InvoiceDate"].max()}")

display(Markdown("\n #### --- Umbrales estadísticos calculados ---"))
print(f"Percentil 75 (límite sin riesgo): {p75:.1f} días de inactividad.")
print(f"Percentil 90 (límite riesgo moderado): {p90:.1f} días de inactividad.")

display(Markdown("\n #### --- Distribución de la variable objetivo ---"))
print(target_df["Target"].value_counts().sort_index())
print("\nPorcentaje por clase:")
print(target_df["Target"].value_counts(normalize=True).sort_index() * 100)

Total de registros limpios: 406829
Rango de fechas completo: 2010-12-01 08:26:00 hasta 2011-12-09 12:50:00



 #### --- División temporal estratégica ---

Ventana de observación (Features): 2010-12-01 08:26:00 a 2011-08-31 17:45:00
Ventana de predicción (Target):   2011-09-01 08:25:00 a 2011-12-09 12:50:00



 #### --- Umbrales estadísticos calculados ---

Percentil 75 (límite sin riesgo): 184.0 días de inactividad.
Percentil 90 (límite riesgo moderado): 282.0 días de inactividad.



 #### --- Distribución de la variable objetivo ---

Target
0    2526
1     499
2     335
Name: count, dtype: int64

Porcentaje por clase:
Target
0    75.178571
1    14.851190
2     9.970238
Name: proportion, dtype: float64


#### Feature Engineering: construcción del perfil de comportamiento del cliente
Para alimentar el modelo predictivo sin incurrir en filtración de datos, transformamos el histórico transaccional de la ventana de observación en una matriz consolidada a nivel de cliente, en lugar de utilizar métricas estáticas, diseñamos variables de alta densidad informativa que capturan los patrones de compra de cada cuenta. Entre las características clave, calculamos el tiempo promedio entre compras (TBP) y su varianza para medir la predictibilidad del ritmo del cliente, el valor de la orden promedio (AOV) para dimensionar su escala financiera, y la diversidad de SKUs junto con la tasa de cancelaciones como indicadores de fidelidad y fricción operativa.

In [ ]:
features_df = pd.DataFrame({"CustomerID": df_obs["CustomerID"].unique()})

finances = df_obs.groupby("CustomerID").agg(
    TotalExpenditure=("Revenue", "sum"),
    TicketAverage=("Revenue", "mean"),
    ExpenditureVariance=("Revenue", "std"),
    TotalQuantities=("Quantity", "sum")
).reset_index()

finances["ExpenditureVariance"] = finances["ExpenditureVariance"].fillna(0)
features_df = pd.merge(features_df, finances, on="CustomerID", how="left")

frecuency = df_obs.groupby("CustomerID")["InvoiceNo"].nunique().reset_index()
frecuency.columns = ["CustomerID", "FrecuencyHist"]
features_df = pd.merge(features_df, frecuency, on="CustomerID", how="left")

skus = df_obs.groupby("CustomerID")["StockCode"].nunique().reset_index()
skus.columns = ["CustomerID", "DiversitySKUs"]
features_df = pd.merge(features_df, skus, on="CustomerID", how="left")

df_obs["Cancellations"] = df_obs["InvoiceNo"].astype(str).str.startswith("C").astype(int)
cancellations = df_obs.groupby("CustomerID")["Cancellations"].mean().reset_index()
cancellations.columns = ["CustomerID", "CancellationsRate"]
features_df = pd.merge(features_df, cancellations, on="CustomerID", how="left")

date_cut_obs = df_obs["InvoiceDate"].max()
recency_obs = df_obs.groupby("CustomerID")["InvoiceDate"].max().reset_index()
recency_obs["RecencyObs"] = (date_cut_obs - recency_obs["InvoiceDate"]).dt.days
features_df = pd.merge(features_df, recency_obs[["CustomerID", "RecencyObs"]], on="CustomerID", how="left")


buys_date = df_obs.groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"].min().reset_index()
buys_date = buys_date.sort_values(by=["CustomerID", "InvoiceDate"])
buys_date["DaysBetweenBuys"] = buys_date.groupby("CustomerID")["InvoiceDate"].diff().dt.days

purchasing_pace = buys_date.groupby("CustomerID").agg(
    AverageTBP=("DaysBetweenBuys", "mean"),
    VarianceTBP=("DaysBetweenBuys", "var")
).reset_index()

purchasing_pace = pd.merge(purchasing_pace, features_df[["CustomerID", "RecencyObs"]], on="CustomerID", how="left")
purchasing_pace["AverageTBP"] = purchasing_pace["AverageTBP"].fillna(purchasing_pace["RecencyObs"] + 1)
purchasing_pace["VarianceTBP"] = purchasing_pace["VarianceTBP"].fillna(0)
purchasing_pace["AverageTBP"] = purchasing_pace["AverageTBP"].replace(0, 0.5)

features_df = pd.merge(features_df, purchasing_pace[["CustomerID", "AverageTBP", "VarianceTBP"]], on="CustomerID", how="left")
features_df["ExcessivePace"] = features_df["RecencyObs"] / features_df["AverageTBP"]

features_df["PurchasingStability"] = np.where(
    features_df["AverageTBP"] > 0,
    np.sqrt(features_df["VarianceTBP"]) / features_df["AverageTBP"],
    0
)

geography = df_obs.groupby("CustomerID")["Country"].first().reset_index()
geography["IsUK"] = (geography["Country"] == "United Kingdom").astype(int)
features_df = pd.merge(features_df, geography[["CustomerID", "IsUK"]], on="CustomerID", how="left")

df_model = pd.merge(features_df, target_df[["CustomerID", "Target"]], on="CustomerID", how="inner")

display(Markdown("### Dataframe procesado (análisis predictivo)"))
display(df_model)

### Dataframe procesado (análisis predictivo)

,CustomerID,TotalExpenditure,TicketAverage,ExpenditureVariance,TotalQuantities,FrecuencyHist,DiversitySKUs,CancellationsRate,RecencyObs,AverageTBP,VarianceTBP,ExcessivePace,PurchasingStability,IsUK,Target
0,17850,5288.63,16.950737,13.603662,1693,35,24,0.048077,202,2.029412,140.029412,99.536232,5.830952,1,2
1,13047,2559.42,16.096981,11.684787,1135,14,84,0.119497,13,19.538462,340.269231,0.665354,0.944106,1,0
2,12583,4214.21,28.668095,20.581401,3462,10,84,0.006803,6,29.333333,449.750000,0.204545,0.722976,0,0
3,13748,580.85,44.680769,57.578315,223,3,11,0.000000,132,70.500000,6384.500000,1.872340,1.133377,1,0
4,15100,635.10,105.850000,215.986263,58,6,1,0.500000,230,8.200000,87.200000,28.048780,1.138792,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3355,14660,128.95,9.210714,11.061525,104,1,14,0.000000,0,1.000000,0.000000,0.000000,0.000000,1,0
3356,13726,292.68,12.725217,4.948352,384,1,23,0.000000,0,1.000000,0.000000,0.000000,0.000000,1,0
3357,15690,111.00,13.875000,13.996186,204,1,8,0.000000,0,1.000000,0.000000,0.000000,0.000000,1,0
3358,17777,194.70,2.905970,2.356764,163,1,63,0.000000,0,1.000000,0.000000,0.000000,0.000000,1,0


#### Decision Tree: nodos que determinan el abandono de un cliente
Dado el desbalanceo inherente de las tres clases (derivado de la segmentación por percentiles), sustituimos la métrica de Accuracy por el Macro F1-Score y la Matriz de Confusión como los indicadores clave de rendimiento, asimismo, configuramos el parámetro de balanceo interno del algoritmo para penalizar proporcionalmente los errores en las clases minoritarias (riesgo moderado y alto), garantizando que el modelo no ignore a los clientes en proceso de abandono. La evaluación se realiza mediante una división de datos estructurada, entrenando con el perfil histórico y validando el rendimiento predictivo final.

#### Capa analítica prescriptiva: modelo de optimización financiera
El sistema evalúa de forma individual a cada cliente clasificado en riesgo moderado o alto utilizando la teoría del valor esperado. Multiplicamos la ganancia histórica del cliente por la probabilidad de abandono estimada por el modelo y por la tasa de éxito de la campaña, restando finalmente el costo de pauta publicitaria. El presupuesto de retención se aprueba única y exclusivamente para las cuentas cuyo resultado sea matemáticamente positivo, garantizando que cada dólar invertido tenga un retorno de inversión esperado favorable.

La decisión comercial se rige formalmente bajo la siguiente ecuación de optimización:

$$VEN = (V_{clie} \times P_{churn} \times E_{camp}) - C_{camp}$$

Donde los parámetros y variables se definen como:

* **$VEN$ (Valor Esperado Neto):** Indicador financiero que determina la viabilidad de la inversión; si $VEN > 0$ la campaña se **aprueba**, si $VEN \le 0$ se **rechaza**.
* **$V_{clie}$ (Ganancia del Cliente):** Variable continua extraída del CRM que representa el gasto total histórico (`TotalExpenditure`) del cliente evaluado.
* **$P_{churn}$ (Probabilidad de Churn):** Probabilidad continua ($0.0$ a $1.0$) arrojada por el método `predict_proba` del modelo CART para las clases de riesgo.
* **$E_{camp}$ (Efectividad de la Campaña):** Constante estimada por el equipo de Marketing fijada en el **30% ($0.30$)**, que representa la probabilidad de éxito de rescate.
* **$C_{camp}$ (Costo de la Campaña):** Inversión publicitaria fija por cliente asignada por el departamento de Marketing, equivalente a **$15.00 USD**.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_model.drop(columns=["CustomerID", "Target"]),
                                                    df_model["Target"],
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df_model["Target"])

cart_model = DecisionTreeClassifier(criterion="entropy",
                                    max_depth=10,
                                    min_samples_leaf=10,
                                    class_weight="balanced",
                                    random_state=42)

cart_model.fit(X_train, y_train)

y_pred = cart_model.predict(X_test)
probabilities = cart_model.predict_proba(X_test)

class_names = ["Clase 0", "Clase 1", "Clase 2"]
classification_report = classification_report(y_test, y_pred, target_names=class_names)
macro_f1 = f1_score(y_test, y_pred, average="macro")
confusion_matrix = confusion_matrix(y_test, y_pred)

display(Markdown("#### --- RESULTADOS DE VALIDACIÓN DEL MODELO ---"))
print(f"Reporte de clasificación por clase:\n {classification_report}")
print(f"Macro F1-Score Global: {macro_f1:.4f}\n")
print(f"Matriz de Confusión (filas: real, columnas: predicho):\n {confusion_matrix}")
print("-" * 70)
if macro_f1 >= 0.7:
    print("Estado: MODELO VALIDADO PARA PRODUCCIÓN (EXCELENTE) - Alta precisión.")
else:
    print("Estado: REVISAR SEGMENTACIÓN - Precisión por debajo del umbral objetivo.")

#### --- RESULTADOS DE VALIDACIÓN DEL MODELO ---

Reporte de clasificación por clase:
               precision    recall  f1-score   support

     Clase 0       0.96      0.82      0.88       505
     Clase 1       0.61      0.86      0.72       100
     Clase 2       0.63      0.99      0.77        67

    accuracy                           0.84       672
   macro avg       0.74      0.89      0.79       672
weighted avg       0.88      0.84      0.85       672

Macro F1-Score Global: 0.7894

Matriz de Confusión (filas: real, columnas: predicho):
 [[412  54  39]
 [ 14  86   0]
 [  1   0  66]]
----------------------------------------------------------------------
Estado: MODELO VALIDADO PARA PRODUCCIÓN (EXCELENTE) - Alta precisión.


#### Herramienta interactiva que analiza las variables de Churn y genera un reporte analítico de asignación presupuestaria

In [ ]:
analysis_roi = X_test.copy()
analysis_roi["RealTarget"] = y_test
analysis_roi["ModelPredict"] = y_pred
analysis_roi["MidProb"] = probabilities[:, 1]
analysis_roi["HighProb"] = probabilities[:, 2]
analysis_roi["TotalRiskProb"] = analysis_roi["MidProb"] + analysis_roi["HighProb"]

CAMPAIGN_COST = 15.0
CAMPAIGN_EFFECTIVENESS = 0.30

analysis_roi["VEN"] = (analysis_roi["TotalExpenditure"] * analysis_roi["TotalRiskProb"] * CAMPAIGN_EFFECTIVENESS) - CAMPAIGN_COST
analysis_roi["CampaignPrescription"] = np.where(analysis_roi["VEN"] > 0, "APROBADO: Desplegar campaña", "RECHAZADO: No invertir")

total_customers_predicted_risk = (analysis_roi["ModelPredict"].isin([1, 2])).sum()
approved_campaigns = (analysis_roi["CampaignPrescription"] == "APROBADO: Desplegar campaña").sum()
budget_savings = (total_customers_predicted_risk - approved_campaigns) * CAMPAIGN_COST

top_5_customers = analysis_roi.loc[(analysis_roi["CampaignPrescription"] == "APROBADO: Desplegar campaña") & (analysis_roi["HighProb"] >= 0.65),:]
top_5_customers = top_5_customers.sort_values(by="VEN", ascending=False).head(5)

app = dash.Dash(__name__)

app.layout = html.Div(id="body", className="e1_body", children=[
html.H1("Análisis prescriptivo de Churn Rate", id="title", className="e1_title"),
html.Div(id="dashboard", className="e1_dashboard", children=[
    html.Div(id="graph_div_1", className="e1_graph_div", children=[
        html.Div(id="dropdown_div_1", className="e1_dropdown_div", style={"justify-content":"flex-start"}, children=[
            dcc.Dropdown(id="dropdown_1", className="e1_dropdown",
                        options = [
                            {"label":"Gasto total","value":"TotalExpenditure"},
                            {"label":"Frecuencia histórica","value":"FrecuencyHist"},
                            {"label":"Tasa de cancelaciones","value":"CancellationsRate"},
                            {"label":"TBT promedio","value":"AverageTBT"}
                        ],
                        value="TotalExpenditure",
                        multi=False,
                        clearable=False)
        ]),
        dcc.Graph(id="histogram", className="e1_graph", figure={})
    ]),
    html.Div(id="graph_div_2", className="e1_graph_div", children=[
        html.Div(id="dropdown_div_2", className="e1_dropdown_div", style={"justify-content":"center"}, children=[
            dcc.Dropdown(id="dropdown_2", className="e1_dropdown", style={"padding-right":"5px"},
                        options = [
                            {"label":"Recencia","value":"RecencyObs"},
                            {"label":"Ritmo excedido","value":"ExcessivePace"},
                            {"label":"TBT promedio","value":"AverageTBT"}
                        ],
                        value="RecencyObs",
                        multi=False,
                        clearable=False),
            dcc.Dropdown(id="dropdown_3", className="e1_dropdown", style={"padding-left":"5px"},
                        options = [
                            {"label":"Gasto total","value":"TotalExpenditure"},
                            {"label":"Ticket promedio","value":"AverageTicket"},
                            {"label":"Cantidades totales","value":"TotalQuantities"}
                        ],
                        value="TotalExpenditure",
                        multi=False,
                        clearable=False)
        ]),
        dcc.Graph(id="scatterplot", className="e1_graph", figure={})
    ]),
]),
   html.H2("Optimización de inversión publicitaria", id="H2", className="e1_title"),
   html.Div(id="div_prescritive", className="e1_div_prescriptive", children=[
       html.Div(f"Total de clientes detectados en riesgo por el modelo: {total_customers_predicted_risk}", id="total_customers", className="e1_txt"),
       html.Div(f"Campañas de pago estratégicamente APROBADAS por ROI: {approved_campaigns}", id="aprove_campaigns", className="e1_txt"),
       html.Div(f"Campañas RECHAZADAS (Se ahorra pauta o pasa a canal gratuito): {total_customers_predicted_risk - approved_campaigns}", id="reject_campagins", className="e1_txt"),
       html.Div(f"Dinero directo RESCATADO / AHORRADO en presupuesto publicitario: ${budget_savings}", id="ROI", className="e1_txt"),
       html.H3("Top 5 clientes a fidelizar", id="H3", style={"font-family":"sans-serif","font-weight":"bold","margin-top":"15px"}),
       html.Div(id="matrix", className="e1_matrix", children=[
            html.Div([html.B("ID", className="e1_header")], id="col_1"),
            html.Div([html.B("Gasto total", className="e1_header")], id="col_2"),
            html.Div([html.B("Probabilidad de Churn", className="e1_header")], id="col_3"),
            html.Div([html.B("Valor Esperado Neto", className="e1_header")], id="col_4"),
            *sum([
                [
                    html.Div(str(row["CustomerID"]), className="e1_cell"),
                    html.Div(f"${row["TotalExpenditure"]:,.2f}", className="e1_cell"),
                    html.Div(f"{row["HighProb"]:.2%}", className="e1_cell"),
                    html.Div(f"${row["VEN"]:,.2f}", className="e1_cell")
                ] for _, row in top_5_customers.iterrows()
            ], [])
       ])
  ])
])


@app.callback(
    [Output(component_id="histogram",component_property="figure"),
    Output(component_id="scatterplot",component_property="figure")],
    [Input(component_id="dropdown_1",component_property="value"),
    Input(component_id="dropdown_2",component_property="value"),
    Input(component_id="dropdown_3",component_property="value")]
)

def update_dashboard(slct_var_histogram, slct_var_X, slct_var_Y):

    histogram = px.histogram(
        df_model,
        x=slct_var_histogram,
        color="Target",
        barmode="group",
        nbins=30,
        title=f"Distribución de clientes por {slct_var_histogram} y nivel de riesgo",
        color_discrete_map={0: "#2ecc71", 1: "#f1c40f", 2: "#e74c3c"},
        labels={"Target": "Estado de riesgo"}
    )

    histogram.update_layout(
        template="plotly_white",
        xaxis_title=slct_var_histogram,
        yaxis_title="Cantidad de clientes (volumen)",
        legend_title="Riesgo real"
    )

    scatterplot = px.scatter(
        analysis_roi,
        x=slct_var_X,
        y=slct_var_Y,
        color="HighProb",
        color_continuous_scale=px.colors.sequential.Reds,
        title=f"Correlación: {slct_var_X} vs {slct_var_Y} (mapeo de probabilidad de Churn)",
        labels={"HighProb": "Probabilidad de Churn"},
        hover_data=[
            slct_var_X,
            slct_var_Y,
            "HighProb",
            "TotalExpenditure",
            "FrecuencyHist",
            "CustomerID"
        ]
    )

    scatterplot.update_traces(hovertemplate=None)

    scatterplot.update_layout(
        template="plotly_white",
        xaxis_title=slct_var_X,
        yaxis_title=slct_var_Y,
        coloraxis_colorbar=dict(title="Probabilidad<br>de Churn", tickformat=".2%")
    )

    return histogram, scatterplot


if __name__ == "__main__":
    app.run(debug=False)